# Stent only

The stent is driven by prescribed displacements, with no artery and no balloon. This is
the cheapest of the three simulation types, and it is the one to run first, because it
exercises the whole chain from the beam mesh to the measured result in seconds rather
than hours.

There are two load cases. Radial expansion drives every centreline node outwards, which
resembles a deployment, and the axial direction is left free, so the foreshortening comes
out as a result instead of being imposed. Axial stretch pulls the two ends apart, which
is the opposite motion, and the stent necks inward as it lengthens. Both cases move the
stent along the same linkage, so they are worth comparing directly.

Every simulation type follows the same four steps:

```
build_input()  ->  check()  ->  run()  ->  postprocess()
```

In [1]:
# "sphinx_gallery" renders each Plotly figure as a self-contained text/html output,
# so the views below also work on the documentation website without a running kernel.
import plotly.io as pio
pio.renderers.default = "sphinx_gallery"

## 1. Check the toolchain

4C runs inside a Docker container, so there is no 4C to install and nothing to compile.
What is needed is Docker itself, about 6 GB of free disk for the image, and Rosetta on
Apple Silicon, because the image is built for amd64 only.

`preflight()` checks all of that before a solve is started. A failing item is named
together with the command that fixes it.

In [2]:
from stentfit.sim import FourCRunner, print_preflight

print_preflight(FourCRunner().preflight())

4C runner preflight  (backend: docker)
  [OK  ] docker cli    /usr/local/bin/docker
  [OK  ] docker daemon server 29.6.2
  [OK  ] host / image  arm64 -> linux/amd64 (emulated, expect it to be slower)
  [OK  ] image         ghcr.io/4c-multiphysics/4c:main present
  [OK  ] digest        sha256:17ac2c98d5d54e18... matches the pinned 2026.3.0-dev build
  => ready


## 2. Load the stent

The stent has to be skeletonised already, which is what `stent_skeleton.ipynb` does.
Everything below is sized from the measurements in that output, so the stent is the
only input.

In [ ]:
from pathlib import Path

from stentfit import Simulation, Stent
from stentfit.sim import StentOnlySettings

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

STENT_NAME = "stent01"
STENT_DIR = REPO / "examples/data/output/stent_skeleton" / STENT_NAME
OUTPUT_DIR = REPO / "examples/data/output/simulation"

stent = Stent.load(str(STENT_DIR), stent_name=STENT_NAME)

## 3. Parameters

Everything that can be changed sits in this one cell. Change a number, re-run from here
down, and the new build lands in its own folder next to the old one. The values below are
the ones that produced the converged runs.

`None` on a few of them means the value the stent already carries, such as the measured
strut thickness.

In [ ]:
# 1. What to simulate
SO_BEAM_MATERIAL   = "elastic"      # "elastic" | "elastoplastic"
SO_CASES           = ["radial_expand", "axial_stretch"]

# 2. Stent material
SO_YOUNGS          = 2.0e5          # MPa; 200 GPa is 316L / CoCr to within a few percent
SO_POISSON         = 0.3
SO_DENSITY         = 0.0            # irrelevant for a static analysis
SO_YIELD_STRENGTH  = 300.0          # MPa; read only when elastoplastic
SO_TANGENT_RATIO   = 64.0 / 380.0   # post-yield tangent modulus, as a fraction of YOUNGS

# 3. Mesh
SO_L_EL_PER_STRUT  = 1.0            # element length, as a multiple of strut thickness
SO_BEAM_CLASS      = "Beam3rHerm2Line3"
SO_STRUT_THICKNESS = stent.stent_features["strut_thickness"]   # mm; None = the measured value

# 4. Load cases
# Engineering strains at t = 1: signed, dimensionless, relative to the undeformed stent.
SO_STRAINS         = {'radial_expand': 0.60, 'axial_stretch': 0.10}
SO_GRIP_FRAC       = 0.2            # axial cases: grip band, as a fraction of the tip ring

# 4b. Self-contact: stops the struts passing through each other
# NOTE: self-contact is not working now, it needs to be fixed.
SO_SELF_CONTACT    = False          # True | False
SO_CONTACT_PENALTY = 0.01           # penalty, as a fraction of E*A / l_el
SO_CONTACT_G0      = 0.05           # force eases in over this many strut thicknesses
SO_CONTACT_EXCLUDE = 0.5            # skip elements this close to a welded crown

# 5. Solver
SO_N_STEPS         = 20           # None -> 20 elastic, 200 plastic
SO_MAX_ITER        = 20           # None -> 20
SO_TOL_RESIDUUM    = 1e-8         # None -> 1e-8
SO_PREDICTOR       = "TangDis"    # None -> "TangDis" elastic, "ConstDis" plastic
SO_LINE_SEARCH     = "Full Step"  # None -> "Full Step" elastic, "Backtrack" plastic

This cell only moves the names above into the settings object the pipeline takes.

In [ ]:
so = StentOnlySettings(
    material=SO_BEAM_MATERIAL, cases=tuple(SO_CASES),
    youngs=SO_YOUNGS, poisson=SO_POISSON, density=SO_DENSITY,
    yield_strength=SO_YIELD_STRENGTH, tangent_modulus_ratio=SO_TANGENT_RATIO,
    beam_class=SO_BEAM_CLASS, l_el_per_strut=SO_L_EL_PER_STRUT,
    strut_thickness=SO_STRUT_THICKNESS,
    strains=SO_STRAINS, grip_frac=SO_GRIP_FRAC,
    self_contact=SO_SELF_CONTACT, contact_penalty_frac=SO_CONTACT_PENALTY,
    contact_g0_per_strut=SO_CONTACT_G0, contact_exclusion_per_strut=SO_CONTACT_EXCLUDE,
    n_steps=SO_N_STEPS, max_iter=SO_MAX_ITER, tol_residuum=SO_TOL_RESIDUUM,
    predictor=SO_PREDICTOR, line_search=SO_LINE_SEARCH)

print(so.material, f"E={so.youngs:g} MPa", f"l_el={so.l_el_per_strut:g}x strut",
      f"{so.n_steps} steps", so.predictor,
      f"self-contact {'on' if so.self_contact else 'off'}")

## 4. Build the input

One folder per load case, each holding the 4C input file, a `.vtu` mesh preview, and a
`run_parameters.yaml` that records every parameter behind it. That record is written now,
at build time, so a run that later fails can still be identified.

Two things are worth reading in the output. The junction weld gap should be about 1e-15 mm,
because anything larger means the wrong beam nodes were welded and the stent will come
apart under load. `check()` reports the beam-to-solid coupling, which this type does not
have, so it says so rather than printing nothing.

In [ ]:
sim = Simulation(stent, sim_type="stent_only", settings=so, output_dir=OUTPUT_DIR)

written = sim.build_input()
sim.check()

for path in written:
    print(f"\n{path.name}")
    for f in sorted(path.parent.iterdir()):
        print(f"   {f.name:38s} {f.stat().st_size / 1e6:7.2f} MB")

## 5. Solve and measure

Solving belongs in a terminal rather than in a notebook cell. 4C streams its convergence
history to the screen, and that is what is needed when a run fails, so it should not
scroll away inside a cell output. Several runs can also go at once, because each one locks
its own folder.

`report` reads the `.pvd` time series back and reduces every step to the numbers that mean
something for a stent: diameter, length, foreshortening, peak strain, and how close the
struts came to yielding. It writes `metrics.csv` and `summary.yaml` next to the results.

In [ ]:
print("run these in a terminal:\n")
for case in SO_CASES:
    print(f"  python -m stentfit.run solve  stent_only {STENT_NAME} {case}")
print()
for case in SO_CASES:
    print(f"  python -m stentfit.run report stent_only {STENT_NAME} {case}")

## 6. Where the files are

```
examples/data/output/simulation/stent_only/<stent>/<case>/
    stent_<case>.4C.yaml     the input 4C solves
    *_mesh.vtu               the mesh preview, for ParaView
    run_parameters.yaml      every parameter that produced this run
    run.log                  the solver output, once it has been run
    out_*/                   4C's own results
    results/metrics.csv      one row per load step
    results/summary.yaml     the headline numbers
```

In ParaView, open `out_*-structure-beams.pvd` for the stent. Do not warp it, because its
points are already deformed. Add **Extract Surface** and then **Tube** to see the struts
with their real thickness.